# PROJECT - Building a binary classification a model that takes a URL and predicts:
# Phishing (1) or Legitimate (0)

Phishing is a type of cyber attack where someone creates a fake website or message to trick you into giving:
Passwords
Credit card details
Personal information

It pretends to be a trusted source.

A legitimate website is a real, official, and trusted site created by the actual company or organization.
It:
Protects your data
Uses secure connections
Does NOT trick you

# LOADING DATASET 

SOURCE - KAGGLE 

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df=pd.read_csv("phishingWebsiteDetection.csv")
df

,index,having_IPhaving_IP_Address,URLURL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,1,-1,1,1,1,-1,-1,-1,-1,-1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,2,1,1,1,1,1,-1,0,1,-1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,3,1,0,1,1,1,-1,-1,-1,-1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,4,1,0,1,1,1,-1,-1,-1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,5,1,0,-1,1,1,-1,1,1,-1,...,-1,1,-1,-1,0,-1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11050,11051,1,-1,1,-1,1,1,1,1,-1,...,-1,-1,1,1,-1,-1,1,1,1,1
11051,11052,-1,1,1,-1,-1,-1,1,-1,-1,...,-1,1,1,1,1,1,1,-1,1,-1
11052,11053,1,-1,1,1,1,-1,1,-1,-1,...,1,1,1,1,1,-1,1,0,1,-1
11053,11054,-1,-1,1,1,1,-1,-1,-1,1,...,-1,1,1,1,1,-1,1,1,1,-1


# DATA UNDERSTANDING

In [13]:
#no. of rows and columns
print(df.shape)

(11055, 32)


In [14]:
#checking missing values
print(df.isnull().sum())
#checking duplicate values
print("Duplicates:", df.duplicated().sum())

index                          0
having_IPhaving_IP_Address     0
URLURL_Length                  0
Shortining_Service             0
having_At_Symbol               0
double_slash_redirecting       0
Prefix_Suffix                  0
having_Sub_Domain              0
SSLfinal_State                 0
Domain_registeration_length    0
Favicon                        0
port                           0
HTTPS_token                    0
Request_URL                    0
URL_of_Anchor                  0
Links_in_tags                  0
SFH                            0
Submitting_to_email            0
Abnormal_URL                   0
Redirect                       0
on_mouseover                   0
RightClick                     0
popUpWidnow                    0
Iframe                         0
age_of_domain                  0
DNSRecord                      0
web_traffic                    0
Page_Rank                      0
Google_Index                   0
Links_pointing_to_page         0
Statistica

No missing or duplicate values in the dataset

In [138]:
X = df.drop("Result", axis=1)
y = df["Result"]

In [141]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [143]:
# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Feature scaling (especially for linear models) - normalizing the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [145]:
from sklearn.preprocessing import StandardScaler
import joblib
joblib.dump(scaler, "scaler.pkl") 

['scaler.pkl']

saved the standard data

In [148]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [150]:
models = {
    "Logistic Regression": LogisticRegression( max_iter=2000,
    solver='lbfgs'),
    "Random Forest": RandomForestClassifier(n_estimators=100,random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier()
}

In [152]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [154]:
results = []

for name, model in models.items():

    # Models that need scaled data
    if name in ["Logistic Regression", "SVM", "KNN"]:

        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)

        y_prob = model.predict_proba(X_test_scaled)[:, 1]

    else:

        # Tree-based models
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        y_prob = model.predict_proba(X_test)[:, 1]

    results.append({

        "Model": name,

        "Accuracy": accuracy_score(y_test, y_pred),

        "Precision": precision_score(
            y_test,
            y_pred
        ),

        "Recall": recall_score(
            y_test,
            y_pred
        ),

        "F1 Score": f1_score(
            y_test,
            y_pred
        ),

        "ROC-AUC": roc_auc_score(
            y_test,
            y_prob
        )

    })

results_df = pd.DataFrame(results)

print(results_df)

                 Model  Accuracy  Precision    Recall  F1 Score   ROC-AUC
0  Logistic Regression  0.923112   0.927502  0.937849  0.932647  0.977092
1        Random Forest  0.967888   0.961778  0.982470  0.972014  0.994164
2    Gradient Boosting  0.947083   0.944531  0.963347  0.953846  0.991741
3                  SVM  0.952510   0.947819  0.969721  0.958645  0.987644
4                  KNN  0.945274   0.948576  0.955378  0.951965  0.985258


In [156]:
import joblib

joblib.dump(models["Random Forest"], "phishing_model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [180]:
sample = X_test.iloc[[100]]

model = models["Random Forest"]
model.fit(X_train, y_train)

prediction = model.predict(sample)

print("Prediction:", prediction)

if prediction[0] == 1:
    print("⚠ Phishing Website")
else:
    print("✅ Legitimate Website")

Prediction: [-1]
✅ Legitimate Website


In [182]:
print("Train Accuracy:", model.score(X_train, y_train))
print("Test Accuracy:", model.score(X_test, y_test))

Train Accuracy: 1.0
Test Accuracy: 0.9678878335594754
